<a href="https://colab.research.google.com/github/manideepnera/ai-meme-generator/blob/reliability-first/ai_meme_image_service.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q fastapi uvicorn nest-asyncio pyngrok pillow torch torchvision diffusers accelerate


In [ ]:
from google.colab import userdata
from pyngrok import ngrok
from pyngrok.conf import PyngrokConfig

# 1. Retrieve token from Colab Secrets
NGROK_AUTH = userdata.get("NGROK_AUTH")

if not NGROK_AUTH:
    raise ValueError("NGROK_AUTH not found in Colab Secrets!")

# 2. Stop existing ngrok process
ngrok.kill()

# 3. Save the correct authentication token
ngrok.set_auth_token(
    NGROK_AUTH,
    pyngrok_config=PyngrokConfig()
)

print("Ngrok authentication configured successfully!")

# 4. Start tunnel (port 8000)
public_url = ngrok.connect(8000)

# 5. Display public URL
print("Public URL:", public_url)

Ngrok authentication configured successfully!
Public URL: NgrokTunnel: "https://windy-vespine-mechanically.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
MODEL_TYPE = "stable_diffusion"
# Options (you control this):
# "stable_diffusion"
# "sdxl"
# "turbo"
# "mock"


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = None

def load_model():
    global pipe

    if MODEL_TYPE == "stable_diffusion":
        from diffusers import StableDiffusionPipeline
        pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch.float16 if device == "cuda" else torch.float32
        ).to(device)

        if device == "cuda":
            pipe.enable_attention_slicing()

        print("✅ Stable Diffusion loaded")

    elif MODEL_TYPE == "sdxl":
        from diffusers import StableDiffusionXLPipeline
        pipe = StableDiffusionXLPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16
        ).to(device)

        print("✅ SDXL loaded")

    elif MODEL_TYPE == "turbo":
        from diffusers import StableDiffusionPipeline
        pipe = StableDiffusionPipeline.from_pretrained(
            "stabilityai/sd-turbo",
            torch_dtype=torch.float16
        ).to(device)

        print("✅ SD Turbo loaded")

    elif MODEL_TYPE == "mock":
        pipe = "mock"
        print("✅ Mock model loaded")

    else:
        raise ValueError("Unknown MODEL_TYPE")

load_model()


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

✅ Stable Diffusion loaded


In [ ]:
from PIL import Image

def generate_image(prompt, negative_prompt):
    """
    THIS FUNCTION IS THE CONTRACT.
    Backend will NEVER be affected.
    """

    if MODEL_TYPE == "mock":
        return Image.new("RGB", (512, 512), color=(30, 30, 30))

    if MODEL_TYPE == "sdxl":
        result = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt
        )
        return result.images[0]

    # Default (SD, Turbo)
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=20,
        guidance_scale=7.5
    )
    return result.images[0]


In [ ]:
from PIL import ImageDraw, ImageFont
import textwrap

def add_caption(image, caption, position):
    draw = ImageDraw.Draw(image)

    try:
        font = ImageFont.truetype("Impact.ttf", size=40)
    except:
        font = ImageFont.load_default()

    w, h = image.size
    max_width = int(w * 0.9)

    # Wrap text
    lines = textwrap.wrap(caption, width=25)
    text = "\n".join(lines)

    # ✅ Pillow-safe text size calculation
    bbox = draw.multiline_textbbox((0, 0), text, font=font, align="center")
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    if position.lower() == "top":
        y = 20
    else:
        y = h - text_h - 20

    x = w // 2

    # Outline for readability
    for dx in [-2, -1, 0, 1, 2]:
        for dy in [-2, -1, 0, 1, 2]:
            draw.multiline_text(
                (x + dx, y + dy),
                text,
                font=font,
                fill="black",
                anchor="ma",
                align="center"
            )

    # Main text
    draw.multiline_text(
        (x, y),
        text,
        font=font,
        fill="white",
        anchor="ma",
        align="center"
    )

    return image


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import base64, io

app = FastAPI(title="Image Generator Service")

class MemeRequest(BaseModel):
    image_prompt: str
    negative_prompt: str
    caption: str
    text_position: str



@app.post("/generate")
def generate(req: MemeRequest):
    image = generate_image(req.image_prompt, req.negative_prompt)
    image = add_caption(image, req.caption, req.text_position)

    buf = io.BytesIO()
    image.save(buf, format="PNG")

    return {
        "image_base64": base64.b64encode(buf.getvalue()).decode()
    }


In [ ]:
public_url = ngrok.connect(8000)
print("🌍 Public URL:", public_url)


🌍 Public URL: NgrokTunnel: "https://windy-vespine-mechanically.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
import nest_asyncio
from uvicorn import Config, Server

nest_asyncio.apply()

server = Server(
    Config(app=app, host="0.0.0.0", port=8000, log_level="info")
)
await server.serve()


INFO:     Started server process [596]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


  0%|          | 0/20 [00:00<?, ?it/s]

INFO:     106.221.188.184:0 - "POST /generate HTTP/1.1" 200 OK


  0%|          | 0/20 [00:00<?, ?it/s]

INFO:     106.221.188.184:0 - "POST /generate HTTP/1.1" 200 OK
